### A5.4.12. Build Compilation Cache Key

**Problem:**

Build a cache key generator that uniquely identifies a compiled computation so the same computation can be looked up and reused without recompilation.

**Explanation:**

A compilation cache avoids recompiling the same graph by storing compiled artifacts keyed by a hash of the graph structure. The key must capture everything that affects the compiled output: operation types, input shapes, dtypes, and graph topology.

How to build it:

1. Walk the graph in topological order.
2. For each node, create a string representation containing the operation type, input shapes, and dtype.
3. Concatenate all node representations in order.
4. Hash the concatenated string (e.g., using SHA-256) to produce a fixed-size cache key.
5. Use this key to look up previously compiled results in a dictionary.

**Example:**

Two calls with the same graph structure and shapes produce the same cache key. Changing a shape produces a different key, triggering recompilation.

In [ ]:
import hashlib


class GraphNode:
    def __init__(self, name, op_type, shape, dtype, inputs=None):
        self.name = name
        self.op_type = op_type
        self.shape = shape
        self.dtype = dtype
        self.inputs = inputs or []


def compute_cache_key(nodes):
    parts = []
    for node in nodes:
        input_names = ",".join(inp.name for inp in node.inputs)
        part = f"{node.op_type}|{node.shape}|{node.dtype}|[{input_names}]"
        parts.append(part)
    signature = ";".join(parts)
    return hashlib.sha256(signature.encode()).hexdigest()[:16]


compilation_cache = {}


def compile_or_lookup(nodes):
    key = compute_cache_key(nodes)
    if key in compilation_cache:
        print(f"  Cache HIT: {key}")
        return compilation_cache[key]
    compiled = f"compiled_program_{key}"
    compilation_cache[key] = compiled
    print(f"  Cache MISS: {key} → compiled")
    return compiled


x1 = GraphNode("x", "input", (32, 784), "float32")
w1 = GraphNode("w", "input", (784, 128), "float32")
mm1 = GraphNode("mm", "matmul", (32, 128), "float32", [x1, w1])

print("First call (shape 32x784):")
compile_or_lookup([x1, w1, mm1])

print("\nSecond call (same shapes):")
compile_or_lookup([x1, w1, mm1])

x2 = GraphNode("x", "input", (64, 784), "float32")
w2 = GraphNode("w", "input", (784, 128), "float32")
mm2 = GraphNode("mm", "matmul", (64, 128), "float32", [x2, w2])

print("\nThird call (shape 64x784):")
compile_or_lookup([x2, w2, mm2])

print(f"\nCache entries: {len(compilation_cache)}")

**References:**

📘 XLA Documentation — [Compilation Caching](https://openxla.org/xla)

📘 JAX Documentation — [JIT Tracing and Caching](https://jax.readthedocs.io/en/latest/jax-101/02-jitting.html)

---

[⬅️ Previous: Build Kernel Dispatch Table](./11_build_kernel_dispatch_table.ipynb) | [Next: Build Minimal Machine Learning Library Architecture ➡️](./13_build_minimal_machine_learning_library_architecture.ipynb)